In [1]:
from blackjack import Environment
from tqdm import tqdm
from collections import defaultdict
import random
import numpy as np

In [2]:
rng = np.random.default_rng(42)
random.seed(42)

In [3]:
def get_state(env: Environment):
    player_card_ranks = []
    for card in env.player_hand:
        player_card_ranks.append(min(card.value, 10))

    state = (
        tuple(sorted(player_card_ranks)),
        min(env.dealer_hand[0].value, 10)
    )

    return state

In [4]:
# Q(s, 0) is stay and Q(s, 1) is hit.
ACTIONS = np.array(["S", "H"])
q_table = defaultdict(lambda: np.zeros(len(ACTIONS), dtype=np.float64))
num_times_visited = defaultdict(lambda: np.zeros(len(ACTIONS), dtype=np.int64))

n_iter = 10_000_000
epsilon = 1.0
min_epsilon = 0.01
epsilon_decay = (min_epsilon / epsilon) ** (1 / n_iter)
gamma = 0.99

In [5]:
training_outcomes = np.zeros(3, dtype=np.int64)  # losses, draws, wins

for _ in tqdm(range(n_iter)):
    env = Environment(five_card_charlie=True)

    while not env.is_over:
        state = get_state(env)

        if rng.random() < epsilon:
            action_index = int(rng.integers(len(ACTIONS)))
        else:
            values = q_table[state]
            best_actions = np.flatnonzero(values == values.max())
            action_index = int(rng.choice(best_actions))

        env.step(ACTIONS[action_index])

        if env.is_over:
            target_q_value = env.score
        else:
            next_state = get_state(env)
            
            # current reward is 0, so bellman's equation simplifies to gamma * max Q(s', a')
            target_q_value = gamma * np.max(q_table[next_state])

        current = q_table[state][action_index]
        
        num_times_visited[state][action_index] += 1
        alpha = 1.0 / num_times_visited[state][action_index]  # schedule learning rate based on number of times a (state, action) pair is visited
        
        q_table[state][action_index] += alpha * (target_q_value - current)

    training_outcomes[env.score + 1] += 1
    epsilon = max(epsilon * epsilon_decay, min_epsilon)

print(f"Learned {len(q_table):,} states.")
print(dict(zip(("losses", "draws", "wins"), training_outcomes)))

  0%|          | 0/10000000 [00:00<?, ?it/s]

100%|██████████| 10000000/10000000 [10:18<00:00, 16156.20it/s]

Learned 5,020 states.
{'losses': np.int64(5089032), 'draws': np.int64(716624), 'wins': np.int64(4194344)}


In [6]:
def choose_greedy_action(env: Environment) -> str:
    """Choose a learned action, breaking ties randomly."""
    values = q_table.get(get_state(env))
    if values is None:
        return str(rng.choice(ACTIONS))

    return str(ACTIONS[int(np.argmax(values))])


def evaluate_agent(n_games: int = 10_000) -> dict[str, float]:
    outcomes = np.zeros(3, dtype=np.int64)

    for _ in range(n_games):
        env = Environment(five_card_charlie=True)
        while not env.is_over:
            env.step(choose_greedy_action(env))
        outcomes[env.score + 1] += 1

    return {
        "loss_rate": outcomes[0] / n_games,
        "draw_rate": outcomes[1] / n_games,
        "win_rate": outcomes[2] / n_games,
        "average_reward": (outcomes[2] - outcomes[0]) / n_games,
    }


evaluate_agent(100_000)

{'loss_rate': np.float64(0.46908),
 'draw_rate': np.float64(0.08192),
 'win_rate': np.float64(0.449),
 'average_reward': np.float64(-0.02008)}

In [7]:
import pickle

with open('bj_q_table.pkl', 'wb') as f:
    pickle.dump(dict(q_table), f)

In [8]:
from itertools import combinations_with_replacement
import pandas as pd


def starting_hand_value(cards: tuple[int, int]) -> tuple[int, bool]:
    total = sum(cards)
    usable_ace = 1 in cards and total + 10 <= 21
    return total + 10 if usable_ace else total, usable_ace


def standard_strategy(cards: tuple[int, int], dealer_upcard: int) -> str:
    """
    The standard blackjack strategy (assuming hit and stand as the only available actions)
    """
    total, is_soft = starting_hand_value(cards)

    if is_soft:
        if total <= 17:
            return "Hit"
        if total == 18:
            return "Stand" if dealer_upcard in range(2, 9) else "Hit"
        return "Stand"

    if total <= 11:
        return "Hit"
    if total == 12:
        return "Stand" if dealer_upcard in range(4, 7) else "Hit"
    if total <= 16:
        return "Stand" if dealer_upcard in range(2, 7) else "Hit"
    return "Stand"


rank_label = {1: "A", **{rank: str(rank) for rank in range(2, 11)}}
comparison_rows = []

for cards in combinations_with_replacement(range(1, 11), 2):
    total, is_soft = starting_hand_value(cards)
    for dealer_upcard in range(1, 11):
        state = (cards, dealer_upcard)
        stand_q, hit_q = q_table.get(state, np.zeros(2))

        if stand_q > hit_q:
            learned_action = "Stand"
        elif hit_q > stand_q:
            learned_action = "Hit"
        else:
            learned_action = "Tie"

        expected_action = standard_strategy(cards, dealer_upcard)
        comparison_rows.append({
            "Player hand": f"{rank_label[cards[0]]},{rank_label[cards[1]]}",
            "Total": total,
            "Soft": is_soft,
            "Dealer": rank_label[dealer_upcard],
            "Q(Stand)": stand_q,
            "Q(Hit)": hit_q,
            "Learned": learned_action,
            "Standard": expected_action,
            "Matches": learned_action == expected_action,
        })

strategy_comparison = pd.DataFrame(comparison_rows)
print(f"Number of disagreeing states: {strategy_comparison.loc[~(strategy_comparison['Matches']), 'Matches'].count()}")

strategy_comparison.sort_values(["Matches", "Soft", "Total", "Player hand", "Dealer"]).head(10)

Number of disagreeing states: 8


,Player hand,Total,Soft,Dealer,Q(Stand),Q(Hit),Learned,Standard,Matches
183,"2,10",12,False,4,-0.234121,-0.175623,Hit,Stand,False
184,"2,10",12,False,5,-0.157699,-0.153956,Hit,Stand,False
185,"2,10",12,False,6,-0.154374,-0.153614,Hit,Stand,False
312,"4,8",12,False,3,-0.200690,-0.235219,Stand,Hit,False
405,"6,6",12,False,6,-0.232432,-0.208792,Hit,Stand,False
261,"3,10",13,False,2,-0.319728,-0.293181,Hit,Stand,False
262,"3,10",13,False,3,-0.281553,-0.277722,Hit,Stand,False
321,"4,9",13,False,2,-0.288440,-0.279604,Hit,Stand,False
109,"2,2",4,False,10,-0.543408,-0.065049,Hit,Hit,True
101,"2,2",4,False,2,-0.290323,0.035576,Hit,Hit,True


In [9]:
n_games = 100_000
outcomes = np.zeros(3, dtype=np.int64)

for _ in range(n_games):
    env = Environment(five_card_charlie=True)
    while not env.is_over:
        player_cards, dealer_card = get_state(env)
        action = "H" if standard_strategy(player_cards, dealer_card) == "Hit" else "S"
        env.step(action)
        
    outcomes[env.score + 1] += 1

print(f"Win-rate: {outcomes[2] / n_games}")

Win-rate: 0.44533
